# Step 5 — Validate the cleaned dataset

**Purpose.** This replaces 05a, 05b and 05c. It verifies the Step 4 output, then checks which inferred column meanings still hold on the cleaned data.

**Input.** Attach the Kaggle Dataset made from `step4_clean_v1.zip`. This notebook reads the cleaned CSV tables and 27 Parquet list parts; it never changes the input.


## Before you run

1. Create a Kaggle Dataset from the Step 4 ZIP, set its visibility as intended, and attach it as input.
2. Run all cells in order. If Kaggle exposes only the ZIP, setup extracts it to the working directory.
3. Use the results to update Step 2 meanings from `[I]` to `[V]` only when the evidence supports them.

Step 4 intentionally stores only analysis-safe list columns. Raw-only checks that need discarded raw list columns are not repeated.


In [ ]:
from __future__ import annotations
import ast, sys, zipfile
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
ROOT = next((x for x in (Path.cwd(), *Path.cwd().parents) if (x / 'environment.yml').exists()), None)

# Preferred path supplied for this public Kaggle Dataset. Kaggle may instead mount it
# as /kaggle/input/myanimelist-clean, so discovery below also checks all input folders.
PREFERRED_INPUT = Path('/kaggle/input/datasets/chuyenhocielts/myanimelist-clean')
REQUIRED_OUTPUT_FILES = {'manifest.csv', 'row_count_log.csv', 'value_cleaning_log.csv',
                         'AnimeList_clean.csv', 'UserList_clean.csv'}
def valid_manifest(path):
    folder = path.parent
    return REQUIRED_OUTPUT_FILES <= {x.name for x in folder.iterdir()} and (folder / 'user_anime_list_parts').is_dir()

roots = [PREFERRED_INPUT, KAGGLE_INPUT, WORKING] if KAGGLE_INPUT.exists() else [ROOT / 'data' / 'processed']
# The preferred directory can be contained in KAGGLE_INPUT, so de-duplicate paths.
manifests = sorted({x.resolve() for root in roots if root and root.exists() for x in root.rglob('manifest.csv') if valid_manifest(x)})
if not manifests and KAGGLE_INPUT.exists():
    archives = list(KAGGLE_INPUT.rglob('step4_clean_v1.zip'))
    if archives:
        with zipfile.ZipFile(archives[0]) as z: z.extractall(WORKING)
        manifests = sorted({x.resolve() for x in WORKING.rglob('manifest.csv') if valid_manifest(x)})
if len(manifests) != 1:
    raise FileNotFoundError(f'Expected exactly one valid Step 4 dataset; found {len(manifests)}: {manifests}')
CLEAN_DIR = manifests[0].parent
LIST_DIR = CLEAN_DIR / 'user_anime_list_parts'
parts = sorted(LIST_DIR.glob('*.parquet'))
if not parts: raise FileNotFoundError(f'No Parquet files in {LIST_DIR}')
pd.set_option('display.width', 220)
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('Dataset:', CLEAN_DIR, '| Parquet parts:', len(parts))


## 1. Confirm the Step 4 output

The manifest and logs must agree with the files before we use them for analysis.


In [ ]:
manifest = pd.read_csv(CLEAN_DIR / 'manifest.csv')
row_log = pd.read_csv(CLEAN_DIR / 'row_count_log.csv')
value_log = pd.read_csv(CLEAN_DIR / 'value_cleaning_log.csv')
na_kwargs = dict(keep_default_na=False, na_values=[''])
anime = pd.read_csv(CLEAN_DIR / 'AnimeList_clean.csv', **na_kwargs)
users = pd.read_csv(CLEAN_DIR / 'UserList_clean.csv', **na_kwargs)
expected = {'username','anime_id','my_status_clean','my_score_clean','my_watched_episodes_clean','my_start_date_clean','my_finish_date_clean','my_start_year_clean','my_finish_year_clean','my_last_updated_clean','my_rewatching_state','my_rewatching_ep_clean','my_tags_clean'}
assert set(pd.read_parquet(parts[0]).columns) == expected
assert (len(anime), len(users), len(parts)) == (14478, 302673, 27)
assert users.username.is_unique
assert not users['username'].isin({'Io_Otonashi', 'Dojin_'}).any()

# Assert row-removal evidence, rather than merely displaying it.
expected_rows = {
    'anime': (14_478, 14_478),
    'users': (302_675, 302_673),
    'lists': (80_076_112, 80_075_053),
}
for table, (before, after) in expected_rows.items():
    record = row_log.loc[row_log['table'].eq(table)]
    assert len(record) == 1, f'Missing or duplicated row-log entry for {table}'
    assert tuple(record[['before_rows', 'after_rows']].iloc[0]) == (before, after), record

# Rules that can be verified wholly from the cleaned anime file and its retained raw columns.
for raw, clean, rule in [
    ('score', 'score_clean', 'anime.score_zero_to_missing'),
    ('rank', 'rank_clean', 'anime.rank_zero_to_missing'),
    ('popularity', 'popularity_clean', 'anime.popularity_zero_to_missing'),
    ('episodes', 'episodes_clean', 'anime.episodes_zero_to_missing'),
]:
    assert not anime.loc[anime[clean].notna(), clean].eq(0).any(), f'Zero remains in {clean}'
    assert int(anime[raw].eq(0).sum()) == int(value_log.set_index('rule').loc[rule, 'count'])
assert anime['low_raters'].equals(anime['scored_by'].lt(100))
assert int(anime['low_raters'].sum()) == int(value_log.set_index('rule').loc['anime.low_raters', 'count'])

display(manifest)
display(row_log)
display(value_log[value_log['count'].gt(0)])


## 2. Anime-level meanings

Ranks should be negatively correlated with their underlying measures. `related` is parsed to confirm that it describes links between titles.


In [ ]:
def spearman(a, b): return a.rank().corr(b.rank())
print('popularity_clean vs members:', round(spearman(anime.popularity_clean, anime.members), 3))
ranked = anime.dropna(subset=['rank_clean','score_clean'])
print('rank_clean vs score_clean:', round(spearman(ranked.rank_clean, ranked.score_clean), 3))
print('scored_by > members:', int((anime.scored_by > anime.members).sum()))
print('favorites > members:', int((anime.favorites > anime.members).sum()))
print(anime.groupby('type', dropna=False).episodes_clean.agg(anime='size', missing=lambda x: int(x.isna().sum()), median='median', max='max').to_string())
def parse_related(x):
    try: v = ast.literal_eval(x)
    except (ValueError, SyntaxError, TypeError): return None
    if v == []: return []
    if not isinstance(v, dict): return None
    return [(r, i.get('mal_id'), i.get('type')) for r, items in v.items() for i in items if isinstance(i, dict)]
parsed = anime.related.map(parse_related)
links = pd.DataFrame({'anime_id':anime.anime_id,'link':parsed}).dropna().explode('link').dropna()
rel = pd.DataFrame(links.link.tolist(), index=links.index, columns=['relation','related_id','related_type'])
rel = pd.concat([links[['anime_id']], rel], axis=1)
rel.related_id = pd.to_numeric(rel.related_id, errors='coerce')
anime_links = rel[rel.related_type.eq('anime')]
print('unparsable related cells:', int(parsed.isna().sum()))
print('linked anime ids found:', f"{anime_links.related_id.isin(anime.anime_id).mean():.1%}")
print(rel.relation.value_counts().head(10).to_string())


## 3. One pass over the cleaned list table

This expensive cell streams all 27 parts once. It calculates sample counts, reconstructed user statistics, rewatch evidence, date coverage and duplicate pairs within each output part.


In [ ]:
ac, rc, ss, sn, ur, ue, us, un, rn, rewatch_episode_sums = (Counter() for _ in range(10))
rows_total = duplicates = positive_rewatch_ep = 0
states, dates = Counter(), Counter()
rule_failures = Counter()
list_usernames, list_anime_ids = set(), set()
episodes_by_anime = anime.set_index('anime_id')['episodes_clean']
valid_status = {1, 2, 3, 4, 6}

for n, path in enumerate(parts, 1):
    df = pd.read_parquet(path)
    rows_total += len(df)
    duplicates += int(df.duplicated(['username','anime_id']).sum())
    list_usernames.update(df['username'].dropna().astype(str))
    list_anime_ids.update(df['anime_id'].dropna().astype(int))
    ac.update(df.anime_id.value_counts().to_dict()); ur.update(df.username.value_counts().to_dict())
    states.update(df.my_rewatching_state.fillna('unknown').value_counts().to_dict())
    for col in ['my_start_date_clean','my_finish_date_clean','my_start_year_clean','my_finish_year_clean']:
        dates[col] += int(df[col].notna().sum())

    # Step 4 list-cleaning rules: count every violation; all must finish at zero.
    rule_failures['invalid_status'] += int((df['my_status_clean'].notna() & ~df['my_status_clean'].isin(valid_status)).sum())
    rule_failures['invalid_score'] += int((df['my_score_clean'].notna() & ~df['my_score_clean'].between(1, 10)).sum())
    rule_failures['watched_sentinel'] += int(df['my_watched_episodes_clean'].eq(65_535).sum())
    rule_failures['rewatch_sentinel'] += int(df['my_rewatching_ep_clean'].eq(16_777_215).sum())
    watched_total = df['anime_id'].map(episodes_by_anime)
    rule_failures['watched_over_total'] += int((df['my_watched_episodes_clean'].notna() & watched_total.notna() & df['my_watched_episodes_clean'].gt(watched_total)).sum())
    start = pd.to_datetime(df['my_start_date_clean'], errors='coerce')
    finish = pd.to_datetime(df['my_finish_date_clean'], errors='coerce')
    lower, upper = pd.Timestamp('2004-11-01'), pd.Timestamp('2018-05-25')
    rule_failures['date_out_of_range'] += int(((start.notna() & ~start.between(lower, upper)) | (finish.notna() & ~finish.between(lower, upper))).sum())
    rule_failures['finish_before_start'] += int((start.notna() & finish.notna() & finish.lt(start)).sum())
    rule_failures['alias_rows'] += int(df['username'].isin({'Io_Otonashi', 'Dojin_'}).sum())

    rated = df.dropna(subset=['my_score_clean']); watched = df.dropna(subset=['my_watched_episodes_clean'])
    rc.update(rated.anime_id.value_counts().to_dict()); ss.update(rated.groupby('anime_id').my_score_clean.sum().astype(float).to_dict()); sn.update(rated.groupby('anime_id').size().to_dict())
    ue.update(watched.groupby('username').my_watched_episodes_clean.sum().astype(float).to_dict()); us.update(rated.groupby('username').my_score_clean.sum().astype(float).to_dict()); un.update(rated.groupby('username').size().to_dict())
    now = df.my_rewatching_state.eq('yes'); rn.update(df.loc[now,'username'].value_counts().to_dict())
    rewatch_ep = df[df.my_rewatching_ep_clean.fillna(0).gt(0)]
    positive_rewatch_ep += len(rewatch_ep); rewatch_episode_sums.update(rewatch_ep.groupby('username').my_rewatching_ep_clean.sum().astype(float).to_dict())
    print(f'part {n:02d}/{len(parts)}: {rows_total:,} rows')

assert rows_total == 80_075_053 and duplicates == 0
print('rewatch states:', dict(states)); print('positive cleaned rewatch episode rows:', positive_rewatch_ep)
print('date coverage:', {k:f'{v/rows_total:.1%}' for k,v in dates.items()})


## 3b. Assert that Step 4 cleaning rules and table relationships hold

These checks use counters collected during the same single list-table pass above. A failure means the processed dataset must be inspected before it is used for a project conclusion.


In [ ]:
assert not any(rule_failures.values()), dict(rule_failures)
assert list_anime_ids <= set(anime['anime_id']), f'Unknown list anime IDs: {sorted(list_anime_ids - set(anime["anime_id"]))[:5]}'
assert list_usernames <= set(users['username']), f'Unknown list usernames: {sorted(list_usernames - set(users["username"]))[:5]}'
expected_has_rows = users['username'].isin(list_usernames)
assert users['has_list_rows'].equals(expected_has_rows), 'has_list_rows does not match the Parquet usernames'
assert set(ur) == list_usernames, 'Per-user row counter does not match usernames found in list parts'
print('All list cleaning-rule checks passed:', dict(rule_failures))
print('D1 cross-table checks passed: every list anime/user exists; has_list_rows agrees with list parts.')


## 4. Compare sample evidence with published and user statistics

High rank correlation supports an ordering, not equal counts: the list is a crawled subset of MyAnimeList users.


In [ ]:
sample = pd.DataFrame({'on_list':pd.Series(ac,dtype=float),'rated':pd.Series(rc,dtype=float),'score_sum':pd.Series(ss,dtype=float),'score_n':pd.Series(sn,dtype=float)})
sample['mean_score_ours'] = sample.score_sum / sample.score_n
c = anime.set_index('anime_id').join(sample, how='left').fillna({'on_list':0,'rated':0})
def cmp(pub, obs, label):
    ok = c[(c[pub]>0)&(c[obs]>0)]; ratio=ok[obs]/ok[pub]
    print(f'{label}: n={len(ok):,} | Spearman={spearman(ok[pub],ok[obs]):.3f} | median ratio={ratio.median():.3f} | middle 90%={ratio.quantile(.05):.3f}–{ratio.quantile(.95):.3f} | ours>published={int((ratio>1).sum())}')
cmp('members','on_list','members vs users listing anime'); cmp('scored_by','rated','scored_by vs users rating anime')
score_ok=c[(c.score_clean>0)&(c.score_n>=30)]; print('score vs list mean: Pearson=',round(score_ok.score_clean.corr(score_ok.mean_score_ours),3),'mean difference=',round((score_ok.mean_score_ours-score_ok.score_clean).mean(),3))
u_calc=pd.DataFrame({'episodes_ours':pd.Series(ue,dtype=float),'score_sum':pd.Series(us,dtype=float),'score_n':pd.Series(un,dtype=float),'rewatching_now_rows':pd.Series(rn,dtype=float),'rewatch_episode_sum':pd.Series(rewatch_episode_sums,dtype=float)})
u_calc['mean_score_ours']=u_calc.score_sum/u_calc.score_n
u=users.set_index('username').join(u_calc,how='inner');
for pub,calc,label in [('stats_episodes_clean','episodes_ours','episodes'),('stats_mean_score_clean','mean_score_ours','mean score')]:
    ok=u[(u[pub]>0)&(u[calc]>0)]; print(f'{label}: n={len(ok):,} | Spearman={spearman(ok[pub],ok[calc]):.3f} | median ours/published={(ok[calc]/ok[pub]).median():.3f}')
r=u.dropna(subset=['stats_rewatched']).fillna({'rewatching_now_rows':0,'rewatch_episode_sum':0}); pos=r.stats_rewatched.gt(0)
print('stats_rewatched > 0:',int(pos.sum()), '| also rewatching now:',int((pos&r.rewatching_now_rows.gt(0)).sum()), '| positive clean rewatch episodes:',int((pos&r.rewatch_episode_sum.gt(0)).sum()))
both=r[pos&r.rewatch_episode_sum.gt(0)]
if len(both): print('rewatch stats vs clean rewatch episodes Spearman:',round(spearman(both.stats_rewatched,both.rewatch_episode_sum),3))


## What to do after the run

- Record the observed correlations, ratios and failed checks in the Step 2 dictionary.
- Upgrade a meaning from `[I]` to `[V]` only when the output supports it.
- Keep the sample limitation in every later result.
- Then we can compare six large consulting questions and use smaller diagnostic questions under each one.
